In [ ]:
import os
import sys
from pathlib import Path
from typing import cast

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.getcwd())

import torch
from omegaconf import DictConfig, OmegaConf

from core.data.mining import load_cache
from core.data.module import AlignData
from core.model.bobert import BobertForAlignment
from core.training.align import find_pretraining_checkpoint, load_pretraining_weights, setup_alignment, train
from core.training.setup import find_latest_checkpoint, find_latest_logger_version, setup_device

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {setup_device()}")
print(f"Working directory: {os.getcwd()}")

config = cast(DictConfig, OmegaConf.load("./config.yaml"))
print(OmegaConf.to_yaml(config))

In [ ]:
cache_path = Path(config.alignment.mining_cache_path)
cache = load_cache(
    cache_path,
    alignment_size=config.alignment.get("alignment_size", None),
    random_seed=config.alignment.get("mining_cache_seed", 42),
)

cache.head(), len(cache)

In [ ]:
datamodule = AlignData(config)
datamodule.prepare_data()
datamodule.setup("fit")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
model = BobertForAlignment.from_config(config, device)

summary = model.get_summary()
print(f"\n--- BERT Encoder Information ---")
print(f"Total Parameters: {summary['trainable_parameters'] / 1e6:.2f}M")
print(f"Model Dimension: {model.bert.d_model}")
print(f"Number of Heads: {model.bert.n_heads}")
print(f"Number of Layers: {model.bert.n_layers}")

In [ ]:
pretrain_checkpoint = find_pretraining_checkpoint(config.pretraining.checkpoint_dir)
stats = load_pretraining_weights(model, pretrain_checkpoint)
print(
    f"loaded={stats['loaded']} skipped={stats['skipped']} "
    f"missing={stats['missing']} unexpected={stats['unexpected']}"
)
print(f"checkpoint={stats['checkpoint_path']}")
print(f"loaded difficulty head: {stats['loaded_difficulty_head']}")

In [ ]:
resume_ckpt = None

if resume_ckpt == "latest":
    resume_checkpoint = find_latest_checkpoint(config.alignment.checkpoint_dir)
    if resume_checkpoint is None:
        raise FileNotFoundError(f"No checkpoint found in {config.alignment.checkpoint_dir}")
elif resume_ckpt:
    resume_checkpoint = Path(resume_ckpt)
else:
    resume_checkpoint = None

logger_version = find_latest_logger_version(config.alignment.checkpoint_dir) if resume_checkpoint is not None else None
module, trainer = setup_alignment(config, model, datamodule.normalizer, logger_version=logger_version)

if resume_checkpoint is not None:
    print(f"Resuming from checkpoint: {resume_checkpoint}")
    if logger_version is not None:
        print(f"Appending logs to: logs/version_{logger_version}")

print(f"\nAlignment setup complete.")
print(f"Total epochs: {config.alignment.num_epochs}")
print(f"Training samples: {len(datamodule.train_dataset)}")
print(f"Validation samples: {len(datamodule.val_dataset)}")

In [ ]:
train(
    module,
    trainer,
    datamodule,
    ckpt_path=str(resume_checkpoint) if resume_checkpoint is not None else None,
)

print("\nBoBERT alignment completed!")